# Chemical Tagging of Star Clusters — t-SNE vs UMAP vs EVoC

Reproduces and extends Kos et al. (2017). The original paper tagged clusters
with **t-SNE** on GALAH abundances. Here we benchmark **t-SNE vs UMAP vs EVoC**
on **SDSS-V DR19 (APOGEE) + Gaia DR3**, over the clusters of Garcia-Dias et al.
(2019) plus the Pleiades.

**Key idea** — stars born together share a chemical fingerprint. Chemical
tagging searches for clustering in the high-dimensional abundance space
(C-space) and matches the groups to known clusters. Kinematics (parallax,
proper motion, radial velocity) are *independent* of abundances, so they
provide a clean ground truth.

This notebook runs a **fast demo**: it restricts the sky to a 30° region around
**M 67** and caps the field sample. The first cells (data prep and the
benchmark) take a few minutes; the whole notebook, including the isochrone
grids and the Gaia cross-match, runs in about ten.

This is the **Jupyter** front end of the same material as the marimo notebook
`notebooks/chemical_tagging.py`. Same library calls, same seeds, same numbers —
the difference is the notebook UI, so you can compare the two.

### Running it

You are inside the workshop container
(`ghcr.io/iaa-so-training/day4-clustering`), so the environment is pinned and
there is nothing to install. From your checkout:

```bash
export IMG=ghcr.io/iaa-so-training/day4-clustering:latest
export DAY4="-v $PWD/data:/app/data -v $PWD/results:/app/results -v $PWD/notebooks:/app/notebooks"

docker run --rm -it $DAY4 $IMG uv run cluster download --all      # catalogue + embeddings, once
docker run --rm -it -p 8888:8888 $DAY4 $IMG \
  uv run --extra jupyter jupyter lab --ip=0.0.0.0 --port=8888 --no-browser --IdentityProvider.token=""
```

Then open this notebook (`notebooks/chemical_tagging.ipynb`) at
http://localhost:8888 and run it top to bottom —
**Kernel → Restart Kernel and Run All Cells**.

### How this differs from the marimo version

- **No reactive dependency graph.** Marimo re-runs the cells that depend on a
  widget you touched; here you run cells yourself (`Shift+Enter`), in order.
  The widget cells below use `ipywidgets.interactive_output`, which re-runs one
  render callback in place when you change a control — the closest equivalent.
- **No `mo.cache`.** This notebook memoises the expensive entry points itself
  (see the setup cell) and still benefits from the same on-disk
  prepared-sample cache (`results/cache/prepared/`, first call ~20 s → ~0.04 s).
- **Same figures, same render budget.** The interactive plots cap the grey
  field at `CLUSTER_PLOT_MAX_POINTS` (default 3000) stars per panel; **every
  member is always drawn**. Only unlabelled field stars are thinned.
- **Matplotlib diagnostics render as static PNGs here.** Run
  `%matplotlib widget` first (ipympl is installed) if you want to pan and zoom
  those live instead.

### Why it can feel slow, even on an idle machine

The heavy cells are **serial by design**, so nothing saturates: reading the
gzipped 1.17 GB catalogue is ~20 s on *one* core, UMAP drops to `n_jobs=1`
whenever a `random_state` is set (it warns about this on every run), and EVoC
builds its tree on one core — only t-SNE uses several. Measured on this 16-core
host: prepare 20.6 s at 1.0 core, UMAP 50 s, EVoC 75 s, t-SNE 43 s at 6.1 cores.

When a widget drives a computation, the work happens in this kernel: watch the
cell's `[*]` marker, not the browser. The memo cell makes a repeated
configuration instant; a genuinely new configuration is a real computation.

## Setup — paths, imports, and the memo for expensive calls

`data/`, `results/` and `data/isochrones/` in the library are relative paths,
and Jupyter starts the kernel in the notebook's own directory
(`/app/notebooks`), so the first thing this cell does is move to the project
root. The `_MEMO` dictionary is the stand-in for marimo's `mo.cache`.

In [ ]:
import os
from pathlib import Path

# The library reads data/, results/, data/isochrones/... relative to the project
# root; Jupyter starts this kernel in notebooks/, so step up once.
_PROJECT_ROOT = Path.cwd()
if not (_PROJECT_ROOT / "data").is_dir() and (_PROJECT_ROOT.parent / "data").is_dir():
    _PROJECT_ROOT = _PROJECT_ROOT.parent
os.chdir(_PROJECT_ROOT)
print(f"project root: {Path.cwd()}")

from cluster import config, seeding
from cluster.baseline import (
    baseline_labels as _baseline_labels,
    confusion_matrix_frame,
    plot_confusion,
    separation_scores,
)
from cluster.benchmark import run_benchmark as _run_benchmark
from cluster.catalog import attach_referee, cluster_panels_cell, hr_cell
from cluster.clusters import CLUSTERS, CLUSTER_BY_NAME
from cluster.data import apply_quality_cuts, load_allstar, prepare as _prepare
from cluster.isochrone import gaia_age_cell, isochrone_cell
from cluster.literature import literature_table
from cluster.plots import abundance_violins, embedding_interactive, method_comparison_bar

import matplotlib.pyplot as plt
import pandas as pd
import plotly.io as pio

# Ship the figure JSON, not a copy of plotly.js per figure: marimo did this
# implicitly, and it is most of the difference between a usable page and a
# multi-megabyte one. (Requires the plotly JupyterLab extension, bundled in the
# workshop image.)
pio.renderers.default = "plotly_mimetype"

# --- the mo.cache stand-in --------------------------------------------------
# Jupyter has no cell cache, so re-running a cell recomputes. These two wrappers
# key on the data identity, the settings and the seeds, so re-running a cell
# whose inputs did not change returns the previous object instantly, while a new
# configuration is always a real computation.
import json as _json

_MEMO: dict[str, object] = {}


def memo_prepare(allstar, settings, clusters, **seeds):
    key = "prepare|" + settings.model_dump_json() + "|" + ",".join(c.name for c in clusters) \
        + "|" + _json.dumps(seeds, sort_keys=True, default=str) + "|" + str(allstar)
    if key not in _MEMO:
        _MEMO[key] = _prepare(allstar, settings, clusters, **seeds)
    return _MEMO[key]


def memo_benchmark(prepared, settings, tag="region"):
    key = f"benchmark|{tag}|{len(prepared.X)}x{prepared.X.shape[1]}|" + settings.model_dump_json()
    if key not in _MEMO:
        _MEMO[key] = _run_benchmark(prepared, settings)
    return _MEMO[key]


def memo_baseline(prepared, settings, *, use_kinematics, min_members):
    key = f"baseline|{use_kinematics}|{min_members}|{len(prepared.X)}|" + settings.model_dump_json()
    if key not in _MEMO:
        _MEMO[key] = _baseline_labels(prepared, settings, use_kinematics=use_kinematics, min_members=min_members)
    return _MEMO[key]


settings = config.Settings()
settings.region_radius_deg = 30.0
settings.cluster_names = ["M 67"]
settings.max_stars = 5_000
seeding.seed_everything(settings.random_state)

print(
    f"FAST={settings.fast}  MAX_STARS={settings.max_stars}  "
    f"SNR_MIN={settings.snr_min}  SEED={settings.random_state}"
)
print(f"REGION={settings.region_radius_deg}°  CLUSTERS={settings.cluster_names}")
print(f"ELEMENTS ({len(settings.elements)}): {settings.elements}")

## 0. Locate the allStar catalogue

The pipeline needs the APOGEE DR19 allStar FITS file (1.17 GB). This cell checks
that it is present and stops with a helpful error if it is not. Fetch it from
your checkout with `docker run --rm -it $DAY4 $IMG uv run cluster download --all`
(adds the embeddings + checkpoints used in §0c), or `uv run cluster download` for
the catalogue alone if you are already in a shell inside the container.

In [ ]:
allstar = Path("data/astraAllStarASPCAP-0.6.0.fits.gz")
if not allstar.exists():
    raise FileNotFoundError(
        f"{allstar} not found.\n"
        "From your checkout run:\n"
        "  docker run --rm -it $DAY4 $IMG uv run cluster download --all\n"
        "(or `uv run cluster download --all` inside the container), then rerun this notebook."
    )
print(f"✓ allStar found: {allstar} ({allstar.stat().st_size / 1e9:.2f} GB)")

## 0b. Paper baseline — re-create Garcia-Dias et al. (2019)

Before pulling M 67 out of the field, the 2019 question: take **only the known
cluster members** (all 23 clusters, no field), cluster them, and ask how well
each star returns to its own cluster — scored with the paper's own metrics
(homogeneity / v-measure / accuracy).

Two feature sets: **abundances alone** (the 2019 setup) and **abundances +
kinematics**. This re-runs the pipeline all-sky (a couple of minutes),
independent of the M 67 demo below.

In [ ]:
baseline_settings = config.Settings()
baseline_settings.max_stars = 0  # keep cluster members only
baseline_clusters = [
    c for c in CLUSTERS
    if c.name in baseline_settings.resolve_cluster_names([c.name for c in CLUSTERS])
]
baseline_prepared = memo_prepare(
    allstar, baseline_settings, baseline_clusters,
    seed_position_radius_deg=config.SEED_POSITION_RADIUS_DEG,
    seed_parallax_frac=config.SEED_PARALLAX_FRAC,
    seed_pm_tol=config.SEED_PM_TOL,
    seed_rv_tol=config.SEED_RV_TOL,
    n_refine_passes=config.N_REFINE_PASSES,
    refine_sigma=config.REFINE_SIGMA,
)

rows = []
cm_frames = {}
for kin, tag in ((False, "chem"), (True, "kin")):
    true, labels = memo_baseline(
        baseline_prepared, baseline_settings,
        use_kinematics=kin, min_members=5,
    )
    for name, pred in labels.items():
        scores = separation_scores(true, pred)
        rows.append({
            "features": "chem+kin" if kin else "chem",
            "method": name,
            "n_stars": int(true.size),
            **scores,
        })
    cm_frames[tag] = {
        "true": true,
        "t-SNE": confusion_matrix_frame(true, labels["t-SNE"]),
    }
baseline_table = pd.DataFrame(rows)
print(f"baseline: {len(baseline_clusters)} clusters, all-sky")

In [ ]:
baseline_table.round(3)

In [ ]:
fig_chem = plot_confusion(cm_frames["chem"]["t-SNE"], None, title="t-SNE — abundances only")
plt.show()

In [ ]:
fig_kin = plot_confusion(cm_frames["kin"]["t-SNE"], None, title="t-SNE — abundances + kinematics")
plt.show()

## 0c. The published spectral latent — same stars, same clusterers

The asset bundle ships a **masked spectral autoencoder**: a network trained on
the raw APOGEE spectra (no abundance labels) whose 256-D latent layer is
exported for every star it covers (`data/embeddings/masked_latent.parquet`,
40 879 stars). That is a genuinely different view of a star — it never sees the
ASPCAP abundances, so a metal-poor globular where the abundances collapse still
has a spectrum.

Comparing two feature sets is only fair on **the same stars**. `head_to_head`
intersects the arms' `APOGEE_ID` sets first, applies the ≥5-member rule to the
intersection, then scores every arm on exactly those stars with the same
clusterers and the same seeds — and hands back `n_stars`, `n_clusters` and the
per-arm losses so you can verify it did.

⏱ This is the slowest cell in the notebook: three seeds × three clusterers × two
arms.

In [ ]:
from cluster.headtohead import Arm, head_to_head, pivot_scores, pivot_with_errors

latent_path = Path("data/embeddings/masked_latent.parquet")
if not latent_path.exists():
    raise FileNotFoundError(
        f"{latent_path} not found.\n"
        "It is part of the asset bundle — from your checkout run\n"
        "  docker run --rm -it $DAY4 $IMG uv run cluster download --assets\n"
        "(or `--all` for the catalogue as well), then rerun this notebook."
    )

arms = [
    Arm(label="abundances (16-d)", embedding_path=None),
    Arm(label="masked AE (256-d)", embedding_path=latent_path,
        notes="self-supervised, from spectra"),
]
h2h = head_to_head(
    baseline_prepared, arms, baseline_settings, min_members=5, seeds=(42, 0, 1),
)
print(f"{h2h.n_stars} stars × {h2h.n_clusters} clusters, on the shared APOGEE_IDs")
print(f"stars lost per arm: {h2h.dropped}")

In [ ]:
pivot_scores(h2h, "homogeneity")

In [ ]:
pivot_with_errors(h2h)  # mean ± std over the three seeds

**How to read it.** Homogeneity per clusterer, same 800 stars and 24 clusters
for both rows:

| method | abundances (16-d) | masked AE (256-d) |
|---|---|---|
| t-SNE | 0.23 | **0.73** |
| UMAP | 0.54 | **0.77** |
| EVoC | 0.46 | **0.69** |

The latent wins on all three — the spectra keep chemical information the 16
abundances do not. Two things worth saying out loud:

- These same-population numbers are *lower* than the ones in
  `docs/spectral_benchmark_results.md`, where each arm was scored on its own
  coverage. Both are in the repo; only this table is apples-to-apples.
- Cluster-only separation (here) and field retrieval (§2, members against the
  Simbad referee) are different questions. The field-retrieval version of the
  spectral arm is Track A in `docs/student_activities.md`.

The same comparison from a shell, with seed error bars and a CSV:

```bash
docker run --rm -it $DAY4 $IMG uv run cluster head-to-head \
    --arm "abundances (16-d)=abundances" \
    --arm "masked AE 256-d=data/embeddings/masked_latent.parquet" \
    --out results/head_to_head.csv
```

## 1. Prepare the data

Load allStar → quality cuts → kinematic membership labels → complete-case 16-D
abundance matrix (standardised). Region mode keeps only stars within 30° of
M 67, mirroring the target paper's per-region approach.

In [ ]:
clusters = [
    c
    for c in CLUSTERS
    if c.name in settings.resolve_cluster_names([c.name for c in CLUSTERS])
]

In [ ]:
prepared = memo_prepare(
    allstar,
    settings,
    clusters,
    seed_position_radius_deg=config.SEED_POSITION_RADIUS_DEG,
    seed_parallax_frac=config.SEED_PARALLAX_FRAC,
    seed_pm_tol=config.SEED_PM_TOL,
    seed_rv_tol=config.SEED_RV_TOL,
    n_refine_passes=config.N_REFINE_PASSES,
    refine_sigma=config.REFINE_SIGMA,
)
# external referee (Simbad catalogue) for the benchmark scores
prepared.df = attach_referee(prepared.df, clusters, settings)
print(f"{prepared.X.shape[0]} stars × {prepared.X.shape[1]} abundances")
print("referee counts:")
print(prepared.df.groupby("referee").size())

## 2. Run the benchmark

- **t-SNE** → 2-D → **HDBSCAN**
- **UMAP** → 2-D → **HDBSCAN**
- **EVoC** → clusters the 16-D vectors directly (embedding + density clustering
  fused)

Scores are measured against the **Simbad catalogue** (the external referee), not
the kinematic labels — see §6. §0c runs the same comparison with the published
spectral latent swapped in for the abundances.

⏱ A few minutes: t-SNE is the largest single stage.

In [ ]:
benchmark = memo_benchmark(prepared, settings, tag="region")

## 3. Macro metrics

Macro-averaged recall and precision per method (clusters with ≥ 1 true member).
**Recall** = fraction of a cluster's true members recovered. **Precision** =
fraction of the predicted group that are true members (purity).

In [ ]:
benchmark.macro()

## 4. Visualise — interactive embedding

Pick a method, then zoom and hover the 2-D embedding coloured by referee
(Simbad) membership. Hovering shows the star id, Teff/logg and its cluster
label. EVoC has no public 2-D projection, so it is drawn on the UMAP canvas.

Changing the dropdown re-renders the figure **in place** — the old figure is
discarded, so the page does not accumulate outputs.

In [ ]:
import ipywidgets as widgets
from IPython.display import Markdown, clear_output, display

embed_method = widgets.Dropdown(options=["t-SNE", "UMAP", "EVoC"], value="UMAP", description="method")
embed_method

In [ ]:
# The picker re-draws this cell in place, and the first draw happens here, so
# "Restart Kernel and Run All Cells" leaves every cell with its output and the
# saved notebook is self-contained.
def _render_embedding(change=None):
    clear_output(wait=True)   # redraw this cell in place; a figure inside an
    # ipywidgets.Output deadlocks a headless (nbconvert) run, so the output
    # goes to the cell's own output area
    display(embedding_interactive(benchmark, embed_method.value))


embed_method.observe(_render_embedding, names="value")
_render_embedding()

## 5. Enriched diagnostics

**Left**: abundance violins comparing M 67 members against field stars —
members should be tighter and chemically distinct.

**Right**: grouped bars of macro recall / precision / kNN purity per method. kNN
purity is the parameter-free chemical-cohesion score (do known members sit
together in the embedding?); EVoC exposes no 2-D projection, so it has no purity
bar.

These two are static PNGs here (marimo rendered them as live widgets). Run
`%matplotlib widget` in a cell first if you want them interactive.

In [ ]:
violins = abundance_violins(
    prepared.df, settings.elements, "M 67",
    random_state=settings.random_state,
)
plt.show()

In [ ]:
bars = method_comparison_bar(benchmark)
plt.show()

## 6. Interactive HR diagrams

Pick a cluster and a membership source, then zoom and hover the Gaia CMD
(absolute G vs BP−RP) and the Kiel diagram (logg vs Teff). Hovering a star shows
its id, Teff/logg, and which of the three sources — **catalogue** (Simbad),
**kinematic**, **combined** — flag it, so you can see where the methods agree
and disagree along the sequence.

In [ ]:
cluster_picker = widgets.Dropdown(options=sorted(CLUSTER_BY_NAME), value="M 67", description="cluster")
method_picker = widgets.RadioButtons(options=["catalog", "kinematic", "combined"], value="combined", description="highlight")
widgets.HBox([cluster_picker, method_picker])

In [ ]:
# load the full catalogue once (heavy: ~20 s first time, then the disk cache)
df_hr = load_allstar(allstar, settings.elements)
df_hr = apply_quality_cuts(df_hr, settings)
print(f"{len(df_hr):,} stars after quality cuts")

In [ ]:
def _render_hr(change=None):
    clear_output(wait=True)   # redraw this cell in place; a figure inside an
    # ipywidgets.Output deadlocks a headless (nbconvert) run, so the output
    # goes to the cell's own output area
    display(hr_cell(df_hr, cluster_picker.value, method_picker.value, settings))


cluster_picker.observe(_render_hr, names="value")
method_picker.observe(_render_hr, names="value")
_render_hr()

## 7. Isochrone fit — a membership-quality proxy

Fit a PARSEC isochrone (ASteCA) to the selected membership source's CMD using
**two colours** — Gaia BP−RP *and* 2MASS J−Ks — which breaks the
age-metallicity degeneracy. Grid chosen per cluster type (solar / metal-poor);
32-walker × 1000-step emcee chain.

Same two pickers as §6. ⏱ Every change re-fits the chain (~1–2 min) — there is
no cache behind this one in either front end.

In [ ]:
def _render_isochrone(change=None):
    clear_output(wait=True)   # redraw this cell in place; a figure inside an
    # ipywidgets.Output deadlocks a headless (nbconvert) run, so the output
    # goes to the cell's own output area
    result = isochrone_cell(   # no marimo -> a figure, or a note as text
        df_hr, cluster_picker.value, method_picker.value, settings,
    )
    if isinstance(result, str):
        display(Markdown(result))
    else:
        display(result)


cluster_picker.observe(_render_isochrone, names="value")
method_picker.observe(_render_isochrone, names="value")
_render_isochrone()

## 8. Gaia-only age fit

APOGEE only sees bright giants, so §7 cannot pin globular ages. This cell
queries **Gaia DR3** directly (full depth to the main sequence), selects members
by proper motion, drops the bright giants + horizontal branch (G ≳ 16 for
globulars), and fits the main-sequence turnoff — which fixes the age.
(metallicity/distance stay degenerate in a single colour.) ⏱ network + emcee.

In [ ]:
def _render_gaia(change=None):
    clear_output(wait=True)   # redraw this cell in place; a figure inside an
    # ipywidgets.Output deadlocks a headless (nbconvert) run, so the output
    # goes to the cell's own output area
    result = gaia_age_cell(cluster_picker.value, settings)
    if isinstance(result, str):
        display(Markdown(result))   # offline note instead of a crash
    else:
        display(result)


cluster_picker.observe(_render_gaia, names="value")
_render_gaia()

## 9. Three views of the cluster

Kiel diagram (logg vs Teff, APOGEE spectroscopy), the 2MASS CMD (K vs J−Ks), and
the Gaia CMD (G vs BP−RP, full depth). Together they show why the fit works: the
Kiel/2MASS views separate giants from dwarfs, and the Gaia view supplies the main
sequence that pins the age. ⏱ the Gaia cross-match hits the network again (cached
under `data/gaia/`).

In [ ]:
def _render_panels(change=None):
    clear_output(wait=True)   # redraw this cell in place; a figure inside an
    # ipywidgets.Output deadlocks a headless (nbconvert) run, so the output
    # goes to the cell's own output area
    display(cluster_panels_cell(df_hr, cluster_picker.value, method_picker.value, settings))


cluster_picker.observe(_render_panels, names="value")
method_picker.observe(_render_panels, names="value")
_render_panels()

## 10. Literature comparison

Accepted parameters (VizieR: Dias for open clusters, Harris 1996/2010 for
globulars; globular ages from Dotter 2010 / Marin-Franch 2009). Compare with the
fitted values in §7/§8 to see where the pipeline is furthest from the literature.

In [ ]:
literature_table(CLUSTERS)

## 11. What to look for

- **Recall vs precision**: small `min_cluster_size` → high recall, low precision
  (field merges in). Tune `config.HDBSCAN` and watch the trade-off.
- **Region mode matters**: the paper ran t-SNE on 30–45° regions, not the whole
  sky. Rerun with a different `settings.region_radius_deg` or
  `settings.cluster_names` and watch recall change (edit the setup cell, then
  re-run it and the cells below).
- **Chemical cohesion (kNN purity)** is the cleaner, parameter-free score: after
  embedding, do known members sit together?
- **Scores are against Simbad** (§2): the catalogue is the referee, so
  recall/precision measure *recovery of literature members*, independent of the
  abundances and kinematics used to find them.
- **Spectra beat abundances where abundances are weak** (§0c): on the same stars
  and the same clusterers, the published masked-AE latent adds +0.2–0.5
  homogeneity. `cluster head-to-head` reproduces the table for any pair of
  feature arms.
- **The combination wins** (§6): `MEMBERSHIP_METHOD="combined"` adds
  chemically-consistent stars to the kinematic core and lifts catalogue recall.
  Flip `settings.membership_method` and watch.
- **Isochrone fit** (§7): two colours (BP−RP + J−Ks) pin the age for open
  clusters (M 67 → 4.4 Gyr ✓). Globulars are different: APOGEE's SNR cut sees
  only red giants (logg ≲ 2.5) — the main sequence/turnoff is too faint — and the
  RGB constrains metallicity/distance but **not age** (age std ≳ 0.9 dex). A real
  survey limitation, not a bug: you need deeper photometry for globular ages.
- The Pleiades group is small in APOGEE (~10 clean members) — the hard, honest
  case. M 67 / NGC 6819 / M 3 are rich — the easy showcase.